# Crime Dataset Builder — CABA Motorcycle Crimes

**Objective:** Build a machine-learning-ready dataset focused on crimes committed using motorcycles in the City of Buenos Aires.

**Main data source:** CABA crime datasets — available at https://data.buenosaires.gob.ar/dataset/delitos

## 0. Imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from scipy.spatial import cKDTree

## 1. Load Crime Data

In [ ]:
# Helper function: find all CSV files whose names contain 'delitos' inside data/
def read_many_csv(folder: str, pattern: str = "delitos") -> pd.DataFrame:
    """Read all CSVs matching *pattern* in *folder* and concatenate them."""
    paths = [p for p in glob.glob(os.path.join(folder, "*.csv")) if pattern in os.path.basename(p)]
    frames = [pd.read_csv(p, na_values="NULL") for p in paths]
    return pd.concat(frames, ignore_index=True)

# Load and concatenate all crime files
delito = read_many_csv("data/")

delito.head()

## 2. Initial Cleaning

In [ ]:
# Drop the 'cantidad' column — not needed for modelling
delito = delito.drop(columns=["cantidad"], errors="ignore")

# Keep only motorcycle crimes that have valid coordinates
delito = delito[delito["uso_moto"] == "SI"].dropna(subset=["latitud", "longitud"])

print(f"Records after filtering: {len(delito):,}")

## 3. Feature Engineering

In [ ]:
# --- Temporal variables ---

# Attach label 'Comuna' to the commune number
delito["comuna"] = "Comuna " + delito["comuna"].astype(str)

# Translate weapon use to English
delito["uso_arma"] = delito["uso_arma"].map({"SI": "yes"}).fillna("no")

# Translate crime subtypes to English
subtipo_map = {
    "Robo total": "Theft",
    "Robo automotor": "Car Theft",
}
delito["subtipo"] = delito["subtipo"].replace(subtipo_map)

# Convert time-slot variable to numeric for calculations
delito["franja"] = pd.to_numeric(delito["franja"], errors="coerce")

# Business cycle: summer holiday months vs. regular workday months
holiday_months = ["ENERO", "FEBRERO", "DICIEMBRE"]
delito["ciclo_laboral"] = delito["mes"].apply(
    lambda m: "Holidays" if m in holiday_months else "Workdays"
)

# Weekend flag
weekend_days = ["SABADO", "DOMINGO"]
delito["fin_de_semana"] = delito["dia"].apply(
    lambda d: "yes" if d in weekend_days else "no"
)

# Night flag: before 7 AM or from 6 PM onwards
delito["noche"] = delito["franja"].apply(
    lambda f: "yes" if (f < 7 or f >= 18) else "no"
)

# Banking hours: weekdays between 10 AM and 6 PM
delito["horario_bancario"] = delito.apply(
    lambda r: "yes" if (r["franja"] >= 10 and r["franja"] < 18 and r["fin_de_semana"] == "no") else "no",
    axis=1,
)

# Parse date and extract day-of-month
delito["fecha"] = pd.to_datetime(delito["fecha"], errors="coerce")
delito["day_month"] = delito["fecha"].dt.day

# Start-of-month flag: first 7 days
delito["principo_mes"] = delito["day_month"].apply(lambda d: "yes" if d <= 7 else "no")

# End-of-month flag: last 7 days (day 24+)
delito["fin_de_mes"] = delito["day_month"].apply(lambda d: "yes" if d >= 24 else "no")

# Translate month names from Spanish to English
month_map = {
    "enero": "January",  "febrero": "February", "marzo": "March",
    "abril": "April",    "mayo": "May",          "junio": "June",
    "julio": "July",     "agosto": "August",     "septiembre": "September",
    "octubre": "October","noviembre": "November","diciembre": "December",
}
delito["mes"] = delito["mes"].str.lower().map(month_map)

# Translate day names from Spanish to English
day_map = {
    "lunes": "Monday",   "martes": "Tuesday",  "miercoles": "Wednesday",
    "jueves": "Thursday","viernes": "Friday",   "sabado": "Saturday",
    "domingo": "Sunday",
}
delito["dia"] = delito["dia"].str.lower().map(day_map)

print("Feature engineering complete.")

## 4. Convert to GeoDataFrame and Spatial Filtering

In [ ]:
# Convert the crime dataframe to a GeoDataFrame using WGS-84 (EPSG:4326)
geometry = gpd.points_from_xy(delito["longitud"], delito["latitud"])
delito = gpd.GeoDataFrame(delito, geometry=geometry, crs="EPSG:4326")

# Load CABA neighbourhood geometries and reproject to match crimes
caba = gpd.read_file("data/barrios.geojson").to_crs(delito.crs).make_valid()

# Keep only crimes that fall within CABA boundaries
delito = delito[delito.geometry.within(caba.unary_union)].copy()

# Drop duplicate records
delito = delito.drop_duplicates()

print(f"Records after spatial filter and deduplication: {len(delito):,}")

## 5. Reproject to POSGAR 2007 (EPSG:5347) for Metric Distance Calculations

In [ ]:
CRS_BA = "EPSG:5347"  # POSGAR 2007 — Argentina 5 (metric CRS)

delito = delito.to_crs(CRS_BA)

## 6. Load Ancillary Spatial Datasets

In [ ]:
# --- Police stations ---
# Source: BA DATA — https://data.buenosaires.gob.ar/dataset/comisarias-policia-ciudad
comisarias = gpd.read_file("data/comisarias_policia.geojson").to_crs(CRS_BA).iloc[:, :2]

# --- Street network ---
# Source: BA DATA — https://data.buenosaires.gob.ar/dataset/calles
callejero = gpd.read_file("data/callejero.geojson")[["nomoficial", "tipo_c", "geometry"]]
callejero = callejero.to_crs(CRS_BA)

# Simplify street types to English categories
def map_street_type(t: str) -> str:
    if pd.isna(t):
        return t
    if "AUTOPISTA" in t:
        return "Highway"
    if "PASAJE" in t or "CALLE" in t:
        return "Street"
    if t in ("BOULEVARD", "AVENIDA"):
        return "Avenue"
    if t == "SENDERO":
        return "Path"
    if t in ("PUENTE", "TÚNEL"):
        return "Other"
    return t

callejero["tipo_c"] = callejero["tipo_c"].apply(map_street_type)

# --- ATMs ---
# Source: BA DATA — https://data.buenosaires.gob.ar/dataset/cajeros-automaticos
cajeros_df = pd.read_csv("data/cajeros-automaticos.csv")
cajeros = gpd.GeoDataFrame(
    cajeros_df,
    geometry=gpd.points_from_xy(cajeros_df["long"], cajeros_df["lat"]),
    crs="EPSG:4326",
).to_crs(CRS_BA)
cajeros = cajeros[cajeros["localidad"] == "CABA"][["banco", "red", "terminales", "geometry"]]

# --- Banks ---
# Source: BA DATA — https://data.buenosaires.gob.ar/dataset/bancos
bancos_df = pd.read_csv("data/bancos.csv", sep=";", encoding="latin1")

# Fix malformed coordinate strings: strip all punctuation, then re-insert the decimal point
for col in ["long", "lat"]:
    bancos_df[col] = bancos_df[col].astype(str).str.replace(r"[^\d]", "", regex=True)

# Re-insert decimal separator at position 2 (replicates R's str_sub assignment)
bancos_df["lat"]  = bancos_df["lat"].apply(lambda x: "34." + x[2:])
bancos_df["long"] = bancos_df["long"].apply(lambda x: "58." + x[2:])

# Convert to float and negate (coordinates are in the southern/western hemisphere)
bancos_df["lat"]  = pd.to_numeric(bancos_df["lat"],  errors="coerce") * -1
bancos_df["long"] = pd.to_numeric(bancos_df["long"], errors="coerce") * -1

bancos = gpd.GeoDataFrame(
    bancos_df,
    geometry=gpd.points_from_xy(bancos_df["long"], bancos_df["lat"]),
    crs="EPSG:4326",
).to_crs(CRS_BA)

print("Ancillary datasets loaded.")

## 7. Nearest-Feature Distance Calculations

We use a **cKDTree** (k-d tree) on the projected coordinates for fast nearest-neighbour lookup — equivalent to `st_nearest_feature` + `st_distance` in R.

In [ ]:
def nearest_distance(origins: gpd.GeoDataFrame, targets: gpd.GeoDataFrame) -> np.ndarray:
    """Return the Euclidean distance (metres, in a metric CRS) from each origin to its nearest target."""
    origin_coords  = np.array([(g.x, g.y) for g in origins.geometry])
    target_coords  = np.array([(g.x, g.y) for g in targets.geometry])
    tree = cKDTree(target_coords)
    distances, _ = tree.query(origin_coords, k=1)
    return distances

def nearest_attrs(origins: gpd.GeoDataFrame, targets: gpd.GeoDataFrame, cols: list) -> pd.DataFrame:
    """Return attribute columns of the nearest target for each origin."""
    origin_coords = np.array([(g.x, g.y) for g in origins.geometry])
    target_coords = np.array([(g.x, g.y) for g in targets.geometry])
    tree = cKDTree(target_coords)
    _, idx = tree.query(origin_coords, k=1)
    return targets[cols].iloc[idx].reset_index(drop=True)

# Distance to nearest police station
delito["dist_comisaria"] = nearest_distance(delito, comisarias)

# Distance to nearest ATM
delito["dist_cajeros"] = nearest_distance(delito, cajeros)

# Distance to nearest bank
delito["dist_bancos"] = nearest_distance(delito, bancos)

# Attributes of the nearest ATM (bank name, network, number of terminals)
nearest_atm = nearest_attrs(delito, cajeros, ["banco", "red", "terminales"])
delito = delito.reset_index(drop=True)
delito[["banco", "red", "terminales"]] = nearest_atm

# Street type of the nearest street segment
nearest_street = nearest_attrs(delito, callejero, ["tipo_c"])
delito["tipo_c"] = nearest_street["tipo_c"].values

print("Distance calculations complete.")

## 8. Distance Threshold Flags

In [ ]:
# Crime within 100 m of an ATM
delito["cajeros_a_100"]      = (delito["dist_cajeros"]    <= 100).map({True: "yes", False: "no"})
# Crime within 200 m of an ATM
delito["cajeros_a_200"]      = (delito["dist_cajeros"]    <= 200).map({True: "yes", False: "no"})
# Crime within 100 m of a police station
delito["comisarias_a_100"]   = (delito["dist_comisaria"]  <= 100).map({True: "yes", False: "no"})
# Crime within 200 m of a police station
delito["comisarias_a_200"]   = (delito["dist_comisaria"]  <= 200).map({True: "yes", False: "no"})
# Crime within 100 m of a bank
delito["bancos_a_100"]       = (delito["dist_bancos"]     <= 100).map({True: "yes", False: "no"})
# Crime within 200 m of a bank
delito["bancos_a_200"]       = (delito["dist_bancos"]     <= 200).map({True: "yes", False: "no"})

## 9. Join Property Prices per Square Metre

In [ ]:
# Source: La Nacion — https://www.lanacion.com.ar/propiedades/...
# Price per square metre (USD) by neighbourhood
precio = pd.read_csv("data/precios_m2_caba.csv")
precio["barrio"] = precio["barrio"].str.upper()

# Remove accents (replicates herramientas::remover_tildes)
import unicodedata
def remove_accents(text: str) -> str:
    if pd.isna(text):
        return text
    return "".join(
        c for c in unicodedata.normalize("NFD", text)
        if unicodedata.category(c) != "Mn"
    )

precio["barrio"] = precio["barrio"].apply(remove_accents)
precio = precio.drop(columns=["unit"], errors="ignore")

# Harmonise neighbourhood names in crimes to match the price dataset
delito["barrio"] = delito["barrio"].str.replace(r"[^\w\s]", "", regex=True)
barrio_fixes = {
    "BOCA": "LA BOCA",
    "VILLA LUGANO": "LUGANO",
    "VILLA SANTA RITA": "SANTA RITA",
}
delito["barrio"] = delito["barrio"].replace(barrio_fixes)

# Left-join price data onto the crime dataset
delito = delito.merge(precio, on="barrio", how="left")

print("Price data joined.")

## 10. Join 2010 Census Tract Data

In [ ]:
# Source: BA DATA — https://data.buenosaires.gob.ar/dataset/informacion-censal-por-radio
radios = (
    gpd.read_file(
        "data/informacion-censal-por-radio-2010/informacion_censal_por_radio_2010_wgs84.shp"
    )
    .make_valid()
    .to_crs(CRS_BA)
)

# Compute area in km², population density, and share of households with unmet basic needs (NBI)
radios["area"] = radios.geometry.area / 1_000_000  # m² → km²
radios["densidad_pob"]    = radios["TOTAL_POB"] / radios["area"]
radios["pct_hogares_nbi"] = radios["H_CON_NBI"] / radios["T_HOGAR"]
# Avoid NaN for tracts with zero population
radios.loc[radios["TOTAL_POB"] == 0, "pct_hogares_nbi"] = 0

radios = radios[["densidad_pob", "pct_hogares_nbi", "geometry"]]

# Spatial join: attach census-tract variables to each crime point
delito = gpd.sjoin(delito, radios, how="left", predicate="within")

print("Census tract data joined.")

## 11. Final Column Selection and Renaming

In [ ]:
# Select and rename only the columns needed for modelling
col_map = {
    "anio"            : "year",
    "mes"             : "month",
    "ciclo_laboral"   : "business_cycle",
    "day_month"       : "day",
    "dia"             : "day_of_week",
    "fin_de_semana"   : "weekend",
    "principo_mes"    : "start_of_month",
    "fin_de_mes"      : "end_of_month",
    "franja"          : "hour",
    "noche"           : "night",
    "horario_bancario": "banking_hours",
    "subtipo"         : "crime_type",
    "uso_arma"        : "weapon_use",
    "tipo_c"          : "street_type",
    "cajeros_a_100"   : "atms_within_100m",
    "cajeros_a_200"   : "atms_within_200m",
    "banco"           : "atm_bank",
    "red"             : "atm_network",
    "bancos_a_100"    : "banks_within_100m",
    "bancos_a_200"    : "banks_within_200m",
    "comisarias_a_100": "police_stations_within_100m",
    "comisarias_a_200": "police_stations_within_200m",
    "value"           : "price_per_sqm_usd",
    "densidad_pob"    : "population_density",
    "pct_hogares_nbi" : "pct_households_nbi",
    "barrio"          : "neighbourhood",
    "comuna"          : "comune",
}

delito = delito[list(col_map.keys())].rename(columns=col_map)

delito.head()

## 12. Export

In [ ]:
# Save the final dataset to CSV — geometry is dropped since the DataFrame is now flat
delito.to_csv("data/crime_data.csv", index=False)

print(f"Saved {len(delito):,} rows to data/crime_data.csv")